# Exploração

**Maj Diego / 2º semestre de 2026**

**Objetivos**

1. Comparar o SLAM 100 com um Scanner homólogo 
2. Utilizar a API informada pelo fabricante


## 1. X120 vs SLAM100

| <a href="media/manuais/X120GO_UserGuide_ENG.pdf">STONEX X120</a>  |  <a href="media/manuais/Slam 100 product manual.pdf">FEIMA ROBOTICS SLAM 100</a>|
|-|-|
|Origem: Italiano<br>Laser Scanning FOV: 270°×360° <br> Camera FOV: 200° ( H ) x 100° ( V )<br> Absolute Accuracy: 5cm  <br> Laser Pulse Repetition Rate: 320kHz <br>Maximum Range: 120m <br>Camera Resolution: 5MP <br>Weight: 1,6Kg <br> Laser channels: 16<br> WIFI Password: 12345678<br>App: GOApp| Origem: Chinês <br>Laser Scanning FOV: 270°×360° <br> Camera FOV: 200° ( H ) x 100° ( V )<br> Absolute Accuracy: 5cm  <br> Laser Pulse Repetition Rate: 320kHz <br>Maximum Range: 120m <br>Camera Resolution: 5MP <br>Weight: 1,6Kg <br> Laser channels: 16<br> WIFI Password: 12345678<br> App: SLAM GO|
|<center><img src="media/imgs/arquivos_raw_x120go.jpeg"></center> | <center><img src="media/imgs/arquivos_raw_slam100.jpg"></center> |
|<center><img src="media/imgs/x120go_home.png"></center> | <center><img src="media/imgs/slamgo_home.jpeg"></center> |
|**4.1 How to use X120GO without application** <br> • Turn on the scanner by pressing the power button for a couple of seconds and wait for the LiDAR head to start rotating.<br> • Place the scanner in a stable spot for initialisation. Press the power button once quickly. The green LED will start flashing indicating that the instrument is acquiring data correctly. <br> • Wait one minute for the instrument to initialise. Make sure that there are no persons or objects moving in front nearby.<br> • After one minute has passed, begin acquisition normally.<br> • If you wish to acquire a control point, place the scanner over a target or recognisable point and stand over this point for at least 10 seconds. After 10 seconds resume scanning normally.<br> • To end the scan, press the power button once quickly. The green LED will stop flashing, indicating the end of data acquisition. | **Device power on** <br> • Press the scanner on key for 3 seconds for a long time, and the status indicator light is always green (the battery is fully charged). Wait for the laser head to start rotating, and then the device starts successfully.<br>  **Start collecting** <br> • Press the scanner on key, and the data acquisition function will be turned on. At this time, the status indicator will turn green (the battery is fully charged) and blink.|
|**API**: Não informado | **API** (https://wiki.feima.cool/en/sdk/slam/http):  <br> • Starting the Device: <br> ••  <code>curl --location --request GET 'http://192.168.10.1:19700/slam/start_work'</code> <br> • Status Check Command: <br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam/get_error_status'</code>  <br> • Stopping the Device:  <br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam/end_work'</code> <br>• Get SSD Info:<br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam2000/get_ssd_info'</code> <br> • Get Real-Time Mapping Status:<br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam/get_mapping_info'</code> <br> • Get Battery State <br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam2000/get_battery_state'</code> <br> • Set Motor Speed <br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam2000/set_motor_speed?speed=[float32]'</code> <br> • Get Motor Status <br> •• <code>curl --location --request GET 'http://192.168.10.1:19700/slam2000/get_motor_board_state'</code>  |
|<center><img src="media/imgs/outdoorstonex.jpeg"><center>|<center><img src="media/imgs/outdoorfeima.jpeg"><center>|

## 2. API

### 2.1. Portas abertas

![alt text](media/imgs/nmap.jpeg)

### 2.2. Porta 19700

In [26]:
"""
Cliente HTTP para o SDK do Feima SLAM (SLAM100/SLAM2000).
Documentacao: https://feima-robotics.github.io/en/slam/http/

Base URL: http://{DeviceIP}:19700
Todos os endpoints sao GET, respondendo em JSON com o envelope tipico:
    {"status": 0|1, "message": null|str, "data": {...}}
(alguns endpoints legados nao seguem o envelope — ver docstrings).

Uso rapido:
    from feima_slam_client import FeimaSlamClient
    c = FeimaSlamClient("192.168.1.100")
    print(c.get_version())
    print(c.get_serial_number())
"""

from __future__ import annotations

import time
from typing import Any, Optional

import requests


class FeimaSlamAPIError(Exception):
    """Levantada quando o dispositivo responde com status != 0."""

    def __init__(self, endpoint: str, status: Any, message: Any):
        self.endpoint = endpoint
        self.status = status
        self.message = message
        super().__init__(f"{endpoint} -> status={status} message={message!r}")


class FeimaSlamClient:
    def __init__(
        self,
        device_ip: str,
        port: int = 19700,
        timeout: float = 5.0,
        session: Optional[requests.Session] = None,
        raise_on_error: bool = False,
    ):
        self.base_url = f"http://{device_ip}:{port}"
        self.timeout = timeout
        self.session = session or requests.Session()
        self.raise_on_error = raise_on_error

    # ---------- infraestrutura interna ----------

    def _get(self, path: str, params: Optional[dict] = None) -> Any:
        url = f"{self.base_url}{path}"
        resp = self.session.get(url, params=params, timeout=self.timeout)

        try:
            data = resp.json()
        except ValueError:
            # alguns endpoints (ex.: /slam/restart) podem devolver corpo vazio
            data = {}
        
        if self.raise_on_error and isinstance(data, dict) and data.get("status", 0) not in (0, None):
            raise FeimaSlamAPIError(path, data.get("status"), data.get("message"))

        try:
            resp.raise_for_status()
        except Exception as e:
            # resp.status_code !=200:  # erro de transporte HTTP (nao confundir com status do envelope)
            data = {"endpoint" : path,
                    "error" : e}

        return data

    # ================= /slam/* =================

    def get_version(self) -> dict:
        """GET /slam/get_version — informacoes de versao."""
        return self._get("/slam/get_version")

    def sync_time(self, timestamp: str, timezone: Optional[str] = None) -> dict:
        """GET /slam/sync_time — sincroniza o relogio do dispositivo.
        timestamp: UTC timestamp (string). timezone: opcional.
        """
        params = {"timestamp": timestamp}
        if timezone is not None:
            params["timezone"] = timezone
        return self._get("/slam/sync_time", params)

    def get_sys_time(self) -> dict:
        """GET /slam/get_sys_time — hora atual do sistema embarcado."""
        return self._get("/slam/get_sys_time")

    def exists_timezone(self, timezone: str) -> dict:
        """GET /slam/exists_timezone — checa se um fuso horario (nome de cidade) existe."""
        return self._get("/slam/exists_timezone", {"timezone": timezone})

    def get_wifi_status(self) -> dict:
        """GET /slam/get_wifi_status — configuracao Wi-Fi atual (pais, banda 2.4G/5G)."""
        return self._get("/slam/get_wifi_status")

    def capture(self, camera: Optional[str] = None) -> dict:
        """GET /slam/capture — tira foto(s); salvas em /mnt/udisk/snap/.
        camera: 'color' ou omitido (todas as cameras).
        """
        params = {"camera": camera} if camera else None
        return self._get("/slam/capture", params)

    def start_work(self) -> dict:
        """GET /slam/start_work — inicia o trabalho/mapeamento."""
        return self._get("/slam/start_work")

    def restart(self) -> dict:
        """GET /slam/restart — reinicia o dispositivo (resposta vazia esperada)."""
        return self._get("/slam/restart")

    def end_work(self) -> dict:
        """GET /slam/end_work — encerra o trabalho/mapeamento."""
        return self._get("/slam/end_work")

    def get_serial_number(self) -> dict:
        """GET /slam/get_serial_number — numero de serie do dispositivo."""
        return self._get("/slam/get_serial_number")

    def get_work_status(self) -> dict:
        """GET /slam/get_work_status — 0: StandBy, 1: Working."""
        return self._get("/slam/get_work_status")

    def get_error_status(self) -> dict:
        """GET /slam/get_error_status — status de erro dos subsistemas."""
        return self._get("/slam/get_error_status")

    def get_mapping_info(self) -> dict:
        """GET /slam/get_mapping_info — status do mapeamento em tempo real.
        state: 0 Unknown, 1 Idle, 2 Initializing, 3 Mapping,
               4 Optimizing, 5 Saving, 6 Completed, 100 Error.
        """
        return self._get("/slam/get_mapping_info")

    def remove_record_point(self, index: str) -> dict:
        """GET /slam/remove_record_point — remove um ponto de controle pelo indice."""
        return self._get("/slam/remove_record_point", {"index": index})

    def get_record_point_list(self) -> dict:
        """GET /slam/get_record_point_list — lista de pontos de controle registrados."""
        return self._get("/slam/get_record_point_list")

    def set_record_point_name(self, index: str, name: str) -> dict:
        """GET /slam/set_record_point_name — renomeia um ponto de controle.
        name: max 32 caracteres (letras/numeros).
        """
        return self._get("/slam/set_record_point_name", {"index": index, "name": name})

    def record_point_with_name(self, name: str, point_type: Optional[str] = None) -> dict:
        """GET /slam/record_point_with_name — registra um ponto de controle com nome.
        type: 1 default, 2 static point, 3 feature point.
        """
        params = {"name": name}
        if point_type is not None:
            params["type"] = point_type
        return self._get("/slam/record_point_with_name", params)

    # ================= /slam2000/* =================

    def set_motor_speed(self, speed: float) -> dict:
        """GET /slam2000/set_motor_speed — velocidade do motor (float32)."""
        return self._get("/slam2000/set_motor_speed", {"speed": speed})

    def set_camera_configs(
        self, control_type: int, cam_type: int, param: Optional[int] = None
    ) -> dict:
        """GET /slam2000/set_camera_configs — configura camera.
        control_type: 1 lente colorida, 2 lente optica.
        cam_type (camType): 1 liga/desliga; 2 captura; 3 modo gravacao;
                             4 compartilhamento de dados; 5 compartilhamento YUV;
                             6 modo rajada; 7 exposicao (0 off,1 50Hz,2 60Hz);
                             8 intervalo de captura (frames/s).
        param: valor associado ao cam_type acima (quando aplicavel).
        """
        params = {"controlType": control_type, "camType": cam_type}
        if param is not None:
            params["param"] = param
        return self._get("/slam2000/set_camera_configs", params)

    def format_storage(self) -> dict:
        """GET /slam2000/format_storage — formata o SSD/armazenamento."""
        return self._get("/slam2000/format_storage")

    def set_run_mode(self, mode: Optional[int] = None) -> dict:
        """GET /slam2000/set_run_mode — modo de operacao.
        mode: 1 Stand mode, 0 Back mode.
        """
        params = {"mode": mode} if mode is not None else None
        return self._get("/slam2000/set_run_mode", params)

    def restart_power(self, action_type: str = "25", param: str = "10") -> dict:
        """GET /slam2000/control_by_type — reinicia a alimentacao (exceto MCU).
        Valores default do exemplo da doc: action_type=25, param=10 (msg_id=30 do protocolo).
        """
        return self._get(
            "/slam2000/control_by_type", {"action_type": action_type, "param": param}
        )

    def set_wifi_country_mode(self, country_code: str, freq: str) -> dict:
        """GET /slam2000/set_wifi_country_mode — configura pais/banda do Wi-Fi.
        freq: '0' 2.4G, '1' 5G (string, conforme doc).
        """
        return self._get(
            "/slam2000/set_wifi_country_mode",
            {"countryCode": country_code, "freq": freq},
        )

    def get_main_board_info(self) -> dict:
        """GET /slam2000/get_main_board_info — versao do hardware (placa principal)."""
        return self._get("/slam2000/get_main_board_info")

    def get_motor_board_state(self) -> dict:
        """GET /slam2000/get_motor_board_state — telemetria da placa do motor
        (tensoes, correntes de fase, rotacao, temperaturas, contagem PPS)."""
        return self._get("/slam2000/get_motor_board_state")

    def get_run_state(self) -> dict:
        """GET /slam2000/get_run_state — runtime do sistema (CPU, memoria, temp, uptime)."""
        return self._get("/slam2000/get_run_state")

    def get_proj_state(self) -> dict:
        """GET /slam2000/get_proj_state — projeto atual (nome, caminho, hora, tipo)."""
        return self._get("/slam2000/get_proj_state")

    def set_proj_state(self, name: Optional[str] = None, path: Optional[str] = None) -> dict:
        """GET /slam2000/set_proj_state — define nome/caminho do projeto atual."""
        params = {}
        if name is not None:
            params["name"] = name
        if path is not None:
            params["path"] = path
        return self._get("/slam2000/set_proj_state", params or None)

    def get_battery_state(self) -> dict:
        """GET /slam2000/get_battery_state — telemetria da bateria (tensao, RSOC, ciclos etc.)."""
        return self._get("/slam2000/get_battery_state")

    def get_camera_info(self) -> dict:
        """GET /slam2000/get_camera_info — estado das cameras optica e colorida."""
        return self._get("/slam2000/get_camera_info")

    def get_ssd_info(self) -> dict:
        """GET /slam2000/get_ssd_info — espaco total/livre/usado do SSD (em MB)."""
        return self._get("/slam2000/get_ssd_info")

    def record_point(self, name: Optional[str] = None) -> dict:
        """GET /slam2000/record_point — registra ponto de controle (variante legada).
        name: max 32 caracteres (letras/underscore).
        """
        params = {"name": name} if name else None
        return self._get("/slam2000/record_point", params)

    # ================= /rtk/* =================

    def set_ntrip_server(self, host: Optional[str] = None, port: Optional[str] = None) -> dict:
        """GET /rtk/set_ntrip_server — configura host/porta do servidor NTRIP."""
        params = {}
        if host is not None:
            params["host"] = host
        if port is not None:
            params["port"] = port
        return self._get("/rtk/set_ntrip_server", params or None)

    def set_ntrip_account(
        self, user: Optional[str] = None, pwd: Optional[str] = None, mntp: Optional[str] = None
    ) -> dict:
        """GET /rtk/set_ntrip_mes — configura usuario/senha/mountpoint NTRIP."""
        params = {}
        if user is not None:
            params["user"] = user
        if pwd is not None:
            params["pwd"] = pwd
        if mntp is not None:
            params["mntp"] = mntp
        return self._get("/rtk/set_ntrip_mes", params or None)

    def get_ntrip_settings(self) -> dict:
        """GET /rtk/get_ntrip_settings — configuracao NTRIP atual (host, mountpoint, user, porta)."""
        return self._get("/rtk/get_ntrip_settings")
   

In [ ]:
# ---------------------------------------------------------------------------
# Exemplo de uso / fluxo tipico de uma sessao de escaneamento
# ---------------------------------------------------------------------------
DEVICE_IP = "192.168.10.1"  # ajuste para o IP real do SLAM100 na rede

client = FeimaSlamClient(DEVICE_IP, raise_on_error=True)

print("Versao:", client.get_version())
print("SN:", client.get_serial_number())
print("Wi-Fi:", client.get_wifi_status())
print("Time:", client.get_sys_time())
print("Work:", client.get_work_status())
print("Erros", client.get_error_status())

print()
print("Bateria:", client.get_battery_state())
print("SSD:", client.get_ssd_info())
print("Motor:", client.get_motor_board_state())
print("Camera:", client.get_camera_info())
print("RAM/CPU:", client.get_run_state())
print("Placa:", client.get_main_board_info())
print("Mapping Info:", client.get_mapping_info())

# Sincroniza hora com o instante atual em UTC (epoch, ms ou s conforme firmware)
# print("Sync time:", client.sync_time(timestamp=str(int(time.time()))))


Versao: {'data': {'fpga': 98, 'lidar': 1467, 'vs1126': 289, 'vs405': 294, 'x8': 1467}, 'message': '', 'status': 0}
SN: {'data': {'serialNo': 'SLAM100242801059'}, 'message': '', 'status': 0}
Wi-Fi: {'data': {'countryCode': 'CN', 'isInit': True, 'shouldLeadWifi': False, 'wifiMode': 0, 'wifiName': 'slam100'}, 'message': '', 'status': 0}
Time: {'data': {'offset': 0, 'time': '2026-09-16 13:18:56 UTC', 'timestamp': 1789564736, 'timezone': 'Etc/UTC'}, 'message': '', 'status': 0}
Work: {'data': {'status': 3}, 'message': '', 'status': 0}
Erros {'data': {'devStatus': '', 'statusCode1': 0, 'statusCode2': 7, 'statusCode3': 1600, 'statusCode4': 255, 'statusCode5': 0}, 'message': '', 'status': 0}

Bateria: {'endpoint': '/slam2000/get_battery_state', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam2000/get_battery_state')}
SSD: {'endpoint': '/slam2000/get_ssd_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam2000/get_s

In [38]:
# Fluxo de mapeamento
print("Start work:", client.start_work())
for _ in range(10):
    info = client.get_mapping_info()
    print("Mapping info:", info)
    if info.get("data", {}).get("state") in (5, 6, 100):
        break
    time.sleep(2)
print("End work:", client.end_work())


Start work: {'data': {'status': 2}, 'message': '', 'status': 0}
Mapping info: {'endpoint': '/slam/get_mapping_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam/get_mapping_info')}
Mapping info: {'endpoint': '/slam/get_mapping_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam/get_mapping_info')}
Mapping info: {'endpoint': '/slam/get_mapping_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam/get_mapping_info')}
Mapping info: {'endpoint': '/slam/get_mapping_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam/get_mapping_info')}
Mapping info: {'endpoint': '/slam/get_mapping_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:19700/slam/get_mapping_info')}
Mapping info: {'endpoint': '/slam/get_mapping_info', 'error': HTTPError('404 Client Error: Not Found for url: http://192.168.10.1:1

In [ ]:
# # Pontos de controle
# print("Record point:", client.record_point_with_name(name="PT01", point_type="2"))
# print("Lista de pontos:", client.get_record_point_list())

# # NTRIP (RTK)
# print("NTRIP settings:", client.get_ntrip_settings())

In [ ]:
import socket, ssl, pprint, time
import random
import struct

default_host = "192.168.10.1" # Change this to your Lidar IP
default_port = 19002 # Change this to your Lidar PTC Port
# default_port = 19003
# default_port = 19004
# default_port = 19700
# default_port = 19802

class PTC:
    def __init__(self, host=default_host, port=default_port):
        self.s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        lp = random.randint(10000, 30000)
        self.s.bind(('0.0.0.0', lp))
        self.s.settimeout(30)
        self.s.connect((host, port))

    def closeSocket(self):
        self.s.shutdown(1)
        # self.s.close()

    def ByteToHex(self, h):
        return ''.join(["%02x" % x for x in h]).strip()

    def read_bytes(self, payload_size):
        chunks = []
        bytes_received = 0
        while bytes_received < payload_size:
            chunk = self.s.recv(payload_size - bytes_received)
            if chunk == b"":
                raise RuntimeError("Socket has been unexpectedly closed")
            chunks.append(chunk)
            bytes_received = bytes_received + len(chunk)

        return b"".join(chunks)

    def sender(self, cmd_code, payload):
        """
        :param cmd_code:
        :param payload: payload should input like '015d010207'
        :return:
        """
        if cmd_code in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e',
                        'f']:
            cmd_code = "0" + str(cmd_code)
        print("payload :")
        print(payload)
        if not payload or payload.upper() == "NONE":
            payload_len = 0
            p = '4774' + str(cmd_code) + "00" + struct.pack('>L', payload_len).hex()
        else:
            payload_len = len(bytes.fromhex(payload))
            p = '4774' + str(cmd_code) + "00" + struct.pack('>L', payload_len).hex() + payload
        data = bytes.fromhex(p)
        self.s.send(data)
        response = self.s.recv(8)
        print("response: ")
        print(response)
        r_cmd = bytes.hex(response[2:3])
        r_returnCode = bytes.hex(response[3:4])
        if bytes.hex(response[4:8]) == "\x00\x00\x00\x00":
            r_length = 0
            response_payload = ""
        else:
            r_length = int(bytes.hex(response[4:8]), 16)
            response_payload = self.read_bytes(r_length)
        print("command is: %s, get return code: %s, return length: %s, \nreturn string:\n%s" % (
            r_cmd, r_returnCode, r_length, response_payload))
        final_response = {
            "response_command": r_cmd,
            "response_return_code": r_returnCode,
            "response_payload_length": r_length,
            "response_payload": response_payload
        }
        return final_response

if __name__ == "__main__":
    ss = PTC()
    print(ss.sender('05', None)) # Example command to 'Get Current Fault Log', change the command code and payload as needed

    # save it as a CSV file
    response = ss.sender('05', None)
    correction_data = response.get("response_payload", b"")
    with open("angle_correction.csv", "wb") as f:
        f.write(correction_data)
    print("The data has been saved as angle_correction.csv.")

    ss.closeSocket()

payload :
None


TimeoutError: timed out

In [ ]:
import socket
import struct
import threading
import csv
import math
import numpy as np

# ============================================================
# CONFIGURAÇÃO
# ============================================================

IP_SLAM = "192.168.10.1"

# Temporariamente usando as portas que você encontrou abertas
POINT_CLOUD_PORT = 19002
POSE_PORT = 19003

DELIMITER = b"#FEIMA#"

# quantidade máxima de frames de nuvem para guardar em memória
MAX_CLOUD_FRAMES = 100


# ============================================================
# AUXILIAR: separa os frames usando #FEIMA#
# ============================================================

def receive_frames(sock):
    buffer = b""

    while True:
        data = sock.recv(65536)

        if not data:
            print("Conexão encerrada.")
            return

        buffer += data

        while DELIMITER in buffer:

            frame, buffer = buffer.split(DELIMITER, 1)

            if frame:
                yield frame


# ============================================================
# DECODIFICAÇÃO DO CABEÇALHO DA NUVEM
#
# Documentação:
#
# uint32_t id
# double timestamp
# uint32_t payloadLen
#
# Como a documentação V1 não informa #pragma pack(1),
# testamos automaticamente algumas possibilidades de padding.
# ============================================================

def decode_cloud_frame(frame):

    candidates = [

        # packed:
        # uint32 + double + uint32
        ("packed-16", "<IdI", 16),

        # alinhamento típico C:
        # uint32 + 4 padding + double + uint32
        ("aligned-20", "<I4xdI", 20),

        # sizeof(struct) possivelmente 24 devido ao padding final
        ("aligned-24", "<I4xdI", 24),
    ]

    for layout_name, fmt, payload_offset in candidates:

        try:

            min_header = struct.calcsize(fmt)

            if len(frame) < min_header:
                continue

            id_, timestamp, payload_len = struct.unpack_from(fmt, frame, 0)

            # Cada ponto = 3 float32
            if payload_len % 12 != 0:
                continue

            # tamanho deve fazer sentido
            if payload_offset + payload_len != len(frame):
                continue

            if not math.isfinite(timestamp):
                continue

            payload = frame[
                payload_offset:
                payload_offset + payload_len
            ]

            points = np.frombuffer(
                payload,
                dtype="<f4"
            ).reshape(-1, 3)

            return {
                "layout": layout_name,
                "id": id_,
                "timestamp": timestamp,
                "payload_len": payload_len,
                "points": points.copy()
            }

        except Exception:
            pass

    return None


# ============================================================
# DECODIFICAÇÃO DA POSE
#
# uint32 id
# double timestamp
#
# float ptX
# float ptY
# float ptZ
#
# float qX
# float qY
# float qZ
# float qW
# ============================================================

def decode_pose_frame(frame):

    candidates = [

        # completamente packed
        ("packed-40", "<Id7f"),

        # padding de 4 bytes antes do double
        ("aligned-44", "<I4xd7f"),

    ]

    for layout_name, fmt in candidates:

        size = struct.calcsize(fmt)

        # aceita também padding final até 48 bytes
        if len(frame) not in (size, 48):
            continue

        try:

            values = struct.unpack_from(fmt, frame, 0)

            (
                id_,
                timestamp,
                x,
                y,
                z,
                qx,
                qy,
                qz,
                qw
            ) = values

            if not math.isfinite(timestamp):
                continue

            # checagem simples do quaternion
            qnorm = math.sqrt(
                qx*qx +
                qy*qy +
                qz*qz +
                qw*qw
            )

            return {
                "layout": layout_name,
                "id": id_,
                "timestamp": timestamp,

                "x": x,
                "y": y,
                "z": z,

                "qx": qx,
                "qy": qy,
                "qz": qz,
                "qw": qw,

                "qnorm": qnorm
            }

        except Exception:
            pass

    return None


# ============================================================
# RECEBER NUVEM
# ============================================================

cloud_frames = []


def point_cloud_receiver():

    print(
        f"\n[NUVEM] Conectando em "
        f"{IP_SLAM}:{POINT_CLOUD_PORT}"
    )

    try:

        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        sock.settimeout(10)

        sock.connect(
            (IP_SLAM, POINT_CLOUD_PORT)
        )

        # depois de conectado deixamos sem timeout curto
        sock.settimeout(None)

        print("[NUVEM] TCP conectado.")

        frame_number = 0

        for frame in receive_frames(sock):

            frame_number += 1

            print(
                f"\n[NUVEM] Frame bruto {frame_number}: "
                f"{len(frame)} bytes"
            )

            result = decode_cloud_frame(frame)

            if result is None:

                print(
                    "[NUVEM] Estrutura NÃO corresponde "
                    "ao protocolo Version 1 conhecido."
                )

                print(
                    "[NUVEM] Primeiros 64 bytes:"
                )

                print(
                    frame[:64].hex(" ")
                )

                continue

            points = result["points"]

            print(
                f"[NUVEM] Layout: {result['layout']}"
            )

            print(
                f"[NUVEM] ID: {result['id']}"
            )

            print(
                f"[NUVEM] Timestamp: "
                f"{result['timestamp']:.6f}"
            )

            print(
                f"[NUVEM] payloadLen: "
                f"{result['payload_len']}"
            )

            print(
                f"[NUVEM] Pontos: {len(points)}"
            )

            if len(points) > 0:

                print(
                    "[NUVEM] Primeiro ponto:",
                    points[0]
                )

                print(
                    "[NUVEM] Min XYZ:",
                    points.min(axis=0)
                )

                print(
                    "[NUVEM] Max XYZ:",
                    points.max(axis=0)
                )

            if len(cloud_frames) < MAX_CLOUD_FRAMES:
                cloud_frames.append(points)

    except Exception as e:

        print(
            "[NUVEM] ERRO:",
            repr(e)
        )


# ============================================================
# RECEBER POSES
# ============================================================

def pose_receiver():

    print(
        f"\n[POSE] Conectando em "
        f"{IP_SLAM}:{POSE_PORT}"
    )

    try:

        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        sock.settimeout(10)

        sock.connect(
            (IP_SLAM, POSE_PORT)
        )

        sock.settimeout(None)

        print("[POSE] TCP conectado.")

        with open(
            "slam100_poses.csv",
            "w",
            newline=""
        ) as f:

            writer = csv.writer(f)

            writer.writerow([
                "id",
                "timestamp",
                "x",
                "y",
                "z",
                "qx",
                "qy",
                "qz",
                "qw",
                "qnorm"
            ])

            frame_number = 0

            for frame in receive_frames(sock):

                frame_number += 1

                print(
                    f"\n[POSE] Frame bruto {frame_number}: "
                    f"{len(frame)} bytes"
                )

                result = decode_pose_frame(frame)

                if result is None:

                    print(
                        "[POSE] Estrutura NÃO corresponde "
                        "ao protocolo Version 1 conhecido."
                    )

                    print(
                        "[POSE] Primeiros bytes:"
                    )

                    print(
                        frame[:64].hex(" ")
                    )

                    continue

                print(
                    f"[POSE] Layout: "
                    f"{result['layout']}"
                )

                print(
                    f"[POSE] ID: "
                    f"{result['id']}"
                )

                print(
                    f"[POSE] Timestamp: "
                    f"{result['timestamp']:.6f}"
                )

                print(
                    "[POSE] XYZ:",
                    result["x"],
                    result["y"],
                    result["z"]
                )

                print(
                    "[POSE] quaternion:",
                    result["qx"],
                    result["qy"],
                    result["qz"],
                    result["qw"]
                )

                print(
                    f"[POSE] |q| = "
                    f"{result['qnorm']:.6f}"
                )

                writer.writerow([
                    result["id"],
                    result["timestamp"],

                    result["x"],
                    result["y"],
                    result["z"],

                    result["qx"],
                    result["qy"],
                    result["qz"],
                    result["qw"],

                    result["qnorm"]
                ])

                f.flush()

    except Exception as e:

        print(
            "[POSE] ERRO:",
            repr(e)
        )


# ============================================================
# SALVAR PLY
# ============================================================

def save_ply(filename, points):

    with open(filename, "wb") as f:

        header = (
            "ply\n"
            "format binary_little_endian 1.0\n"
            f"element vertex {len(points)}\n"
            "property float x\n"
            "property float y\n"
            "property float z\n"
            "end_header\n"
        )

        f.write(header.encode("ascii"))

        points.astype(
            "<f4"
        ).tofile(f)


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    cloud_thread = threading.Thread(
        target=point_cloud_receiver,
        daemon=True
    )

    pose_thread = threading.Thread(
        target=pose_receiver,
        daemon=True
    )

    cloud_thread.start()
    pose_thread.start()

    try:

        while True:

            cloud_thread.join(timeout=1)
            pose_thread.join(timeout=1)

            if (
                not cloud_thread.is_alive()
                and
                not pose_thread.is_alive()
            ):
                break

    except KeyboardInterrupt:

        print("\nInterrompido pelo usuário.")

    # --------------------------------------------------------
    # junta os frames de nuvem recebidos
    # --------------------------------------------------------

    if cloud_frames:

        all_points = np.vstack(
            cloud_frames
        )

        print(
            "\nSalvando:",
            len(all_points),
            "pontos"
        )

        save_ply(
            "slam100_cloud.ply",
            all_points
        )

        print(
            "Arquivo criado: slam100_cloud.ply"
        )
        

    print(
        "Arquivo de poses: slam100_poses.csv"
    )